### Ein einfaches Spiel als Observable
Siehe `game.py`

In [ ]:
from game import Game


def callback(event, data):
    print(f'event={event}, data={data}')


def test_game(game):
    game.new_game()
    for i in range(4):
        old_pos = (1, 0)
        new_pos = (i, i)
        game.place(old_pos)
        game.move(old_pos, new_pos)

    for i in range(16):
        row, col = divmod(i, 4)
        game.place((col, row))

    return game

In [ ]:
game = Game()
game

In [ ]:
game = Game()
game.register_callback(callback)
game = test_game(game)

### Gridhelper nun als Klasse
Siehe gridhelper.py und `GridHelper.ipynb`

In [ ]:
%run gridhelper

### Eine View für Game (MultiCanvas mit einem Layer)
Die `__init__`-Methode der BaseView erledigt bereits folgendes:
- Ein MultiCanvas-Objekt wird erstellt und
  als Attribut `mcanvas` gespeichert. Der oberste Layer ist `canvas`.
- Defaultmässig wird auch ein Output-Widget erstellt, in welches Fehlermeldungen und umgeleitet werden.  
- Eine (noch nicht implementierte) Funktion `update` wird als Callback beim Game registriert.  
  `update` muss in der erbenden Klasse überschreiben werden.

Die BaseView hat folgende Methoden:  
- `display`: stellt die dar (wird automatisch aufgrufen, wenn `view` im Juptyerlab dargestellt werden soll).
- `log(msg)`: Falls mit `View(debug=True)` erstellt, wird `msg` ins Output-Widget geleitet.

In [ ]:
from model_view_controller import BaseView
from gridhelper import GridHelper
from ipycanvas import hold_canvas


class View1(BaseView):
    def __init__(self, game, width=100, height=100, nlayers=1, debug=True):
        super().__init__(game, width, height, nlayers, debug)

        self.gridhelper = GridHelper(10, 10, 20, 20, 4, 4)
        self.gridhelper.draw_grid(self.canvas, line_width=2, color='blue')

        self.log('Drawing the Grid')

    def update(self, event, data):
        self.log(f'running update(event={event}, data={data})')
        with hold_canvas():
            self.canvas.clear()
            self.gridhelper.draw_grid(self.canvas, line_width=2, color='blue')
            for pos in self.game.placed:
                self.gridhelper.fill_circle(self.canvas, pos, color='red')


game.remove_callbacks()  # verwenden bestehenes Game-Objekt, loeschen alte Callbacks
view = View1(game)
view

In [ ]:
game.new_game()
old_pos = (1, 0)
new_pos = (1, 1)
game.place(old_pos)

In [ ]:
game.move(old_pos, new_pos)

### Eine weitere View für Game (mit 3 Layern)

In [ ]:
class View2(BaseView):
    def __init__(self, game, width=100, height=100, nlayers=3, debug=True):
        super().__init__(game, width, height, nlayers, debug)

        self.bg, self.fg, self.info = self.mcanvas  # erstellt von __init__ von BaseView

        self.gridhelper = GridHelper(10, 18, 20, 20, 4, 4)
        self.draw_grid()
        self.log('Drawing the Grid')

    def draw_grid(self):
        self.mcanvas.clear()
        self.gridhelper.draw_grid(self.fg, line_width=2, color='blue')

    def place(self, pos):
        self.info.clear()
        self.gridhelper.fill_circle(self.bg, pos, color='red')
        self.gridhelper.stroke_circle(self.info, pos, line_width=3, color='orange')

    def move(self, old_pos, new_pos):
        self.info.clear()
        self.gridhelper.clear_rect(self.bg, old_pos)
        self.gridhelper.fill_circle(self.bg, new_pos, color='red')
        self.gridhelper.stroke_circle(self.info, new_pos, line_width=3, color='orange')

    def update(self, event, data):
        self.log(f'running update(event={event}, data={data})')

        if event == 'new_game':
            self.draw_grid()

        if event == 'place' and data:
            pos, success = data
            self.place(pos)
            if success:
                self.info.fill_text('Congrats!', 10, 13)

        if event == 'move' and data:
            old_pos, new_pos = data
            self.move(old_pos, new_pos)


game.remove_callbacks()
view = View2(game)
view

In [ ]:
game.new_game()
old_pos = (1, 0)
new_pos = (1, 1)
game.place(old_pos)

In [ ]:
game.move(old_pos, new_pos)

In [ ]:
game.place(old_pos)

### Der Controller
`controller = Controller(game, view, callbacks, key_handler=None)` erstellt einen Controller.
Der Key-Handler ist optional.

Der Dict `callbacks` enthält die Callbacks.  

- Callbacks für Tastendrücke werden ohne Arumente aufgerufen.
- Ein Callback `f` für ein Mausevents wird mit den Argumenten `f(controller, x, y, state)` aufgerufen.
- Die Key-Handler Funktion `f` wird mit den Argumenten `f(controller, key, state)` aufgerufen.

`state` ist dabei ein Dict, den die verschiedenen Callbacks zum Kommunizieren nutzen können.  
Dieser Dict ist `controller._state`.

Im Debug-Modus zeigt der Controller an, welche Taste gedrückt wird, und
welches Callback mit welchen Arumenten aufgerufen wird.

In [ ]:
from model_view_controller import Controller


view = View2(game)
controller = Controller(game, view, callbacks=[], key_handler=None)
controller

In [ ]:
game = Game()


def key_handler(self, key, state):
    '''beim Aufruf ist self der controller, key der Tastenname und state ein Dict'''
    pass


def on_mouse_down(self, x, y, state):
    pass


def on_mouse_up(self, x, y, state):
    pass


callbacks = {'n': game.new_game,
             'mouse_down':  on_mouse_down,
             'mouse_up':  on_mouse_up,
             }



view = View2(game)
controller = Controller(game, view, callbacks=callbacks, key_handler=key_handler)
controller

In [ ]:
test_game(game)